<a href="https://colab.research.google.com/github/icsl-aist/hsr-genesis/blob/main/examples/tutorials/4_gripper_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HSR ハンド・グリッパ制御 / HSR Hand & Gripper Control

**目的 / Objective:**

- HSRのハンド（2本指グリッパ）の開閉を学ぶ / Learn to open and close the HSR hand (two-finger gripper).
- 位置制御による開閉と、力制御による把持（grasp）の違いを理解する / Understand the difference between position-controlled opening/closing and force-controlled grasping.
- 箱を把持して持ち上げる一連の動作を体験する / Experience the sequence of grasping and lifting a box.

> **Note:** Run this on a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU* or better).

## Setup / セットアップ

Run the cell below to install dependencies, clone the repo, and configure GPU rendering.
If you run into issues, see the [troubleshoot notebook](7_troubleshoot_colab.ipynb).

下のセルを実行して、依存パッケージのインストール、リポジトリのクローン、GPU レンダリングの設定を行います。
問題が発生した場合は[トラブルシューティングノートブック](7_troubleshoot_colab.ipynb)を参照してください。


In [ ]:
import importlib, urllib.request

# Fetch standalone bootstrap from GitHub (zero hsr_genesis imports).
exec(urllib.request.urlopen(
    "https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/colab_setup.py"
).read())

# One-call setup: installs deps, clones repo (with submodules),
# configures EGL for headless GPU rendering. Safe to re-run.
# See the troubleshoot notebook if anything goes wrong.
setup_colab()


In [ ]:
from hsr_genesis.tutorial_utils import *

init_sim()


## 5. HSRのハンドについて / About the HSR hand

HSRのハンドは **2本指の平行グリッパ** で、1つのモータ関節 (`hand_motor_joint`) で両指が同時に開閉します。

The HSR hand is a **two-finger parallel gripper** driven by a single motor joint (`hand_motor_joint`); both fingers move together.

- `move_hand(v)`: 位置制御。`v=1.0` で全開、`v=0.0` で全閉 / Position control: `v=1.0` fully open, `v=0.0` fully closed.
- `grasp_object(effort)`: 力制御。指定した effort [N] で指を閉じ続け、物体を把持し続ける / Force control: closes the fingers with the given effort [N] and keeps grasping.
- `release_object()`: 把持を解除しハンドを開く / Releases the grasp and opens the hand.

まずはアームをニュートラル姿勢にして、グリッパが見えやすい状態にします。
First, move the arm to a neutral pose so the gripper is easy to see.

## 6. アームをニュートラルへ / Move arm to neutral

In [ ]:
move_arm_neutral()
run(2.0)
show_video()

## 7. ハンドを開く / Open the hand

`move_hand(1.0)` でハンドを全開にします。
Open the hand fully with `move_hand(1.0)`.

In [ ]:
move_hand(1.0)
run(1.0)
show_video()

## 8. ハンドを閉じる / Close the hand

`move_hand(0.0)` でハンドを全閉にします（物体を持っていない状態）。
Close the hand fully with `move_hand(0.0)` (with no object in the hand).

In [ ]:
move_hand(0.0)
run(1.0)
show_video()

## 9. 力制御による把持 / Force-controlled grasping

位置制御 (`move_hand`) では指は指定した位置で止まりますが、`grasp_object(effort)` は **力制御** で指を閉じ続けます。これにより、物体を挟んだ状態で把持力を維持できます。

With position control (`move_hand`) the fingers stop at the commanded position, but `grasp_object(effort)` keeps closing the fingers under **force control**, maintaining a gripping force once an object is between them.

把持対象として赤い箱を配置します。
Spawn a red box as the grasp target.

## 10. 箱を配置 / Spawn a box

In [ ]:
init_sim()
cube = spawn_box((0.5, 0.0, 0.02), color=(0.8, 0.2, 0.2, 1.0))
run(0.5)
show_video()

## 11. 把持位置へ移動 / Move to grasp position

IKでハンドを箱の上に近づけます。ロール 180° で指が下を向くようにします。
Use IK to bring the hand above the box. A roll of 180° points the fingers downward.

In [ ]:
move_hand(1.0)
move_wholebody_ik(0.5, 0.0, 0.02, 180, 0, 0)
run(3.0)
show_video()

## 12. 把持 / Grasp

In [ ]:
grasp_object(5.0)
run(3.0)
show_video()

## 13. 持ち上げる / Lift

把持したままIKでハンドを持ち上げます。`grasp_object` は `run()` 中も力制御が継続します。
Lift the hand with IK while keeping the grasp. `grasp_object` stays active during `run()`.

In [ ]:
move_wholebody_ik(0.5, 0.0, 0.3, 180, 0, 0)
run(2.0)
show_video()

## 14. 離す / Release

In [ ]:
release_object()
run(1.0)
show_video()

## まとめ / Summary

- `move_hand(v)`: 位置制御でハンドを開閉（`1.0`=開, `0.0`=閉） / Position-controlled open/close (`1.0`=open, `0.0`=closed).
- `grasp_object(effort)`: 力制御で把持を維持。`run()` 中も有効 / Force-controlled grasp, stays active during `run()`.
- `release_object()`: 把持を解除してハンドを開く / Releases the grasp and opens the hand.
- IK (`move_wholebody_ik`) と組み合わせて、把持・持ち上げ・離すができる / Combined with IK (`move_wholebody_ik`) you can grasp, lift, and release.

次はチュートリアル5「把持の全シーケンス」で、ベース移動も組み合わせたピック＆プレースに挑戦します。
Next, tutorial 5 "Full grasp sequence" combines base motion for a complete pick-and-place.